# 📤 Notebook 04 — Model Evaluation & Export
## TourMate AI · Tourism Recommendation System

> **Objective:** Final comprehensive evaluation of every model and recommendation component.  
> Produce production-ready artifacts, benchmark all systems, and document the complete ML pipeline.

| Step | Description |
|---|---|
| 1 | Load tuned models & metadata |
| 2 | Classification — full metrics, confusion matrix, ROC curves |
| 3 | Regression — residual analysis, error distribution, feature importance |
| 4 | Recommender — qualitative & quantitative evaluation |
| 5 | Clustering — profile validation, PCA, silhouette |
| 6 | Model comparison dashboard |
| 7 | Export all production artifacts |
| 8 | Final system architecture summary |

---

In [ ]:
# ─── 0. SETUP ──────────────────────────────────────────────────────────────────
import sys, os, json, joblib, time, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, auc,
    mean_squared_error, r2_score, mean_absolute_error,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy.stats as scipy_stats

PALETTE = {'sky':'#0ea5e9','sky_d':'#0284c7','sky_l':'#e0f2fe',
           'mint':'#52c27a','sand':'#d4a373','snow':'#f1f5f9',
           'ink':'#0f172a','muted':'#64748b'}
CHART_COLORS = ['#0ea5e9','#52c27a','#d4a373','#f97316','#8b5cf6','#ec4899','#14b8a6','#ef4444']
plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False,
                     'font.family':'DejaVu Sans'})

print("✓ All imports successful")
print(f"  NumPy {np.__version__} | Pandas {pd.__version__}")


In [ ]:
# ─── 1. LOAD EVERYTHING ────────────────────────────────────────────────────────
MODELS_DIR = '../models'
DATA_PATH  = '../data/tourism_recommendation_dataset_en.csv'

from src.data_preprocessing import load_dataset, clean_dataset, build_attraction_catalog
from src.recommender import (PopularityRecommender, ContentBasedRecommender,
                              HybridRecommender, ActivityRecommender)

# Load raw data
df = load_dataset(DATA_PATH)
df = clean_dataset(df)
catalog = build_attraction_catalog(df)

# Load saved model artifacts
models = {}
for fname in os.listdir(MODELS_DIR):
    if fname.endswith('.pkl'):
        key = fname.replace('.pkl', '')
        try:
            models[key] = joblib.load(os.path.join(MODELS_DIR, fname))
            print(f"  ✓ Loaded: {fname}")
        except Exception as e:
            print(f"  ⚠ Failed: {fname} — {e}")

# Load JSON metadata
for fname in ['model_metrics.json', 'model_metadata.json', 'feature_names.json']:
    path = os.path.join(MODELS_DIR, fname)
    if os.path.exists(path):
        with open(path) as f:
            models[fname.replace('.json','')] = json.load(f)

print(f"\n✓ Loaded {len([k for k in models if not isinstance(models[k], dict)])} model objects")
print(f"  Attractions in catalog: {len(catalog)}")
print(f"  Training records      : {len(df):,}")


In [ ]:
# ─── 2. FEATURE RECONSTRUCTION ────────────────────────────────────────────────
enc_cols = ['attraction_category','season','gender','age_group','attraction_level','province']
for col in enc_cols:
    le = LabelEncoder()
    df[col+'_enc'] = le.fit_transform(df[col].astype(str))

FEATURE_COLS = ['age','ticket_price','visit_duration_hours','spend_amount',
                'other_spend','trip_days','is_holiday_enc','is_group_tour_enc',
                'attraction_category_enc','season_enc','gender_enc',
                'age_group_enc','attraction_level_enc','province_enc']
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
X = df[FEATURE_COLS].fillna(0)

le_target = LabelEncoder()
y_clf = le_target.fit_transform(df['recommendation_level'].astype(str))
y_reg = df['rating'].values

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_clf, test_size=0.20, random_state=42, stratify=y_clf)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.20, random_state=42)

print(f"Feature matrix : {X.shape[0]:,} × {X.shape[1]}")
print(f"Train split    : {X_train_c.shape[0]:,} | Test: {X_test_c.shape[0]:,}")
print(f"Target classes : {le_target.classes_}")


## 1. Classification Model — Final Evaluation

In [ ]:
# ─── 2.1 CLASSIFICATION: FULL METRICS SUITE ──────────────────────────────────
clf = models.get('classification_model')
y_pred_clf = clf.predict(X_test_c)
y_prob_clf = clf.predict_proba(X_test_c)

acc   = accuracy_score(y_test_c, y_pred_clf)
f1_w  = f1_score(y_test_c, y_pred_clf, average='weighted')
f1_m  = f1_score(y_test_c, y_pred_clf, average='macro')
prec  = precision_score(y_test_c, y_pred_clf, average='weighted')
rec   = recall_score(y_test_c, y_pred_clf, average='weighted')

# Cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1  = cross_val_score(clf, X, y_clf, cv=skf, scoring='f1_weighted', n_jobs=-1)
cv_acc = cross_val_score(clf, X, y_clf, cv=skf, scoring='accuracy', n_jobs=-1)

print("=" * 65)
print("  CLASSIFICATION MODEL — FINAL EVALUATION REPORT")
print("  Model: Random Forest Classifier")
print("=" * 65)
print(f"  Test Set Metrics:")
print(f"    Accuracy          : {acc:.4f}")
print(f"    F1-Weighted       : {f1_w:.4f}")
print(f"    F1-Macro          : {f1_m:.4f}")
print(f"    Precision-Weighted: {prec:.4f}")
print(f"    Recall-Weighted   : {rec:.4f}")
print(f"  Cross-Validation (5-fold):")
print(f"    CV F1-W Mean      : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")
print(f"    CV Acc Mean       : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
print("=" * 65)

print("\nPer-Class Report:")
print(classification_report(y_test_c, y_pred_clf,
                             target_names=le_target.classes_, digits=4))


In [ ]:
# ─── 2.2 CONFUSION MATRIX + ROC CURVES + FEATURE IMPORTANCE ─────────────────
fig = plt.figure(figsize=(22, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)

# ── Confusion Matrix (normalized)
ax_cm = fig.add_subplot(gs[0, 0])
cm = confusion_matrix(y_test_c, y_pred_clf)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = ax_cm.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax_cm, shrink=0.8)
for i in range(len(le_target.classes_)):
    for j in range(len(le_target.classes_)):
        ax_cm.text(j, i, f'{cm_norm[i,j]:.3f}\n({cm[i,j]:,})',
                   ha='center', va='center', fontsize=9,
                   color='white' if cm_norm[i,j] > 0.5 else 'black')
ax_cm.set_xticks(range(len(le_target.classes_)))
ax_cm.set_yticks(range(len(le_target.classes_)))
ax_cm.set_xticklabels(le_target.classes_, rotation=20, ha='right', fontsize=9)
ax_cm.set_yticklabels(le_target.classes_, fontsize=9)
ax_cm.set_xlabel('Predicted'); ax_cm.set_ylabel('Actual')
ax_cm.set_title('Confusion Matrix (Normalized)', fontsize=12, fontweight='bold')

# ── ROC Curves (One-vs-Rest)
ax_roc = fig.add_subplot(gs[0, 1])
y_bin = label_binarize(y_test_c, classes=range(len(le_target.classes_)))
for i, (cls, color) in enumerate(zip(le_target.classes_, CHART_COLORS)):
    if y_bin.shape[1] > 1:
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob_clf[:, i])
        auc_val = auc(fpr, tpr)
        ax_roc.plot(fpr, tpr, color=color, lw=2,
                    label=f'{cls[:12]} (AUC={auc_val:.3f})')
ax_roc.plot([0,1],[0,1],'k--',lw=1,label='Random (AUC=0.500)')
ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curves — One-vs-Rest', fontsize=12, fontweight='bold')
ax_roc.legend(fontsize=8, loc='lower right')
ax_roc.grid(alpha=0.3)

# ── Feature Importance
ax_fi = fig.add_subplot(gs[0, 2])
fi = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
colors_fi = [PALETTE['sky'] if v >= fi.quantile(0.75) else
             PALETTE['sky_l'] for v in fi.values]
ax_fi.barh(range(len(fi)), fi.values, color=colors_fi, edgecolor='white')
ax_fi.set_yticks(range(len(fi)))
ax_fi.set_yticklabels(fi.index, fontsize=8.5)
ax_fi.set_xlabel('Gini Importance')
ax_fi.set_title('Feature Importance Ranking', fontsize=12, fontweight='bold')
for i, v in enumerate(fi.values):
    ax_fi.text(v+0.001, i, f'{v:.4f}', va='center', fontsize=8)

# ── Per-Class F1 Bar
ax_cls = fig.add_subplot(gs[1, 0])
from sklearn.metrics import f1_score as f1s
per_class_f1 = [f1s(y_test_c == i, y_pred_clf == i, zero_division=0)
                for i in range(len(le_target.classes_))]
ax_cls.bar(le_target.classes_, per_class_f1,
           color=CHART_COLORS[:len(le_target.classes_)], edgecolor='white')
ax_cls.axhline(f1_w, color=PALETTE['sand'], ls='--', lw=2, label=f'Weighted avg={f1_w:.3f}')
ax_cls.set_ylabel('F1 Score'); ax_cls.set_title('Per-Class F1 Score', fontsize=12, fontweight='bold')
ax_cls.set_xticklabels(le_target.classes_, rotation=20, ha='right', fontsize=9)
ax_cls.legend(fontsize=9)
for i, v in enumerate(per_class_f1):
    ax_cls.text(i, v+0.005, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')

# ── CV Score Distribution
ax_cv = fig.add_subplot(gs[1, 1])
ax_cv.boxplot([cv_f1, cv_acc], patch_artist=True, notch=False,
              boxprops=dict(facecolor=PALETTE['sky_l'], color=PALETTE['sky_d']),
              medianprops=dict(color=PALETTE['sand'], linewidth=2.5),
              labels=['CV F1-Weighted', 'CV Accuracy'])
ax_cv.set_ylabel('Score')
ax_cv.set_title('5-Fold Cross-Validation Distribution', fontsize=12, fontweight='bold')
for i, scores in enumerate([cv_f1, cv_acc], 1):
    ax_cv.text(i, scores.mean(), f'μ={scores.mean():.4f}\nσ={scores.std():.4f}',
               ha='center', va='bottom', fontsize=9,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# ── Prediction Confidence Histogram
ax_conf = fig.add_subplot(gs[1, 2])
max_probs = y_prob_clf.max(axis=1)
correct   = y_pred_clf == y_test_c
ax_conf.hist(max_probs[correct],  bins=30, color=PALETTE['mint'], alpha=0.65,
             label=f'Correct ({correct.sum():,})', density=True)
ax_conf.hist(max_probs[~correct], bins=30, color=PALETTE['sand'], alpha=0.65,
             label=f'Incorrect ({(~correct).sum():,})', density=True)
ax_conf.set_xlabel('Prediction Confidence (max probability)')
ax_conf.set_ylabel('Density')
ax_conf.set_title('Confidence Distribution\n(Correct vs Incorrect Predictions)',
                  fontsize=12, fontweight='bold')
ax_conf.legend(fontsize=9)
ax_conf.axvline(0.5, color='red', ls='--', lw=1.5)

plt.suptitle('Classification Model — Comprehensive Evaluation Dashboard',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('../static/charts/nb4_clf_evaluation.png', bbox_inches='tight', dpi=130)
plt.show()


## 2. Regression Model — Final Evaluation

In [ ]:
# ─── 3.1 REGRESSION: COMPREHENSIVE METRICS ────────────────────────────────────
reg = models.get('regression_model')
y_pred_reg = reg.predict(X_test_r)

r2   = r2_score(y_test_r, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_reg))
mae  = mean_absolute_error(y_test_r, y_pred_reg)
residuals = y_test_r - y_pred_reg

# Additional metrics
mape = np.mean(np.abs(residuals / (y_test_r + 1e-9))) * 100
within_05 = np.mean(np.abs(residuals) <= 0.5) * 100
within_10 = np.mean(np.abs(residuals) <= 1.0) * 100

# Cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2   = cross_val_score(reg, X, y_reg, cv=kf, scoring='r2',          n_jobs=-1)
cv_rmse = cross_val_score(reg, X, y_reg, cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1)

# Normality of residuals
stat_sw, p_sw = scipy_stats.shapiro(residuals[:2000])

print("=" * 65)
print("  REGRESSION MODEL — FINAL EVALUATION REPORT")
print("  Model: Gradient Boosting Regressor")
print("  Target: attraction rating (1–5 scale)")
print("=" * 65)
print(f"  Test Set Metrics:")
print(f"    R² Score          : {r2:.4f}  ({r2*100:.2f}% variance explained)")
print(f"    RMSE              : {rmse:.4f}")
print(f"    MAE               : {mae:.4f}")
print(f"    MAPE              : {mape:.2f}%")
print(f"    Within ±0.5 rating: {within_05:.2f}%")
print(f"    Within ±1.0 rating: {within_10:.2f}%")
print(f"  Cross-Validation (5-fold):")
print(f"    CV R² Mean        : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"    CV RMSE Mean      : {-cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")
print(f"  Residual Normality (Shapiro-Wilk):")
print(f"    W-stat={stat_sw:.4f}, p={p_sw:.4f}")
print(f"    Distribution: {'NOT normal (expected for large n)' if p_sw < 0.05 else 'Normal'}")
print("=" * 65)


In [ ]:
# ─── 3.2 REGRESSION EVALUATION DASHBOARD ─────────────────────────────────────
fig = plt.figure(figsize=(22, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)

# ── Pred vs Actual scatter (colored by error magnitude)
ax1 = fig.add_subplot(gs[0, 0])
err_mag = np.abs(residuals)
scatter = ax1.scatter(y_test_r[:5000], y_pred_reg[:5000],
                       c=err_mag[:5000], cmap='RdYlGn_r', alpha=0.35,
                       s=12, edgecolors='none', vmin=0, vmax=1.5)
plt.colorbar(scatter, ax=ax1, label='|Residual|', shrink=0.8)
mn, mx = y_test_r.min(), y_test_r.max()
ax1.plot([mn,mx],[mn,mx],'b--',lw=2,label='Perfect fit')
ax1.plot([mn,mx],[mn+0.5,mx+0.5],'--',color='gray',lw=1,alpha=0.5,label='±0.5 band')
ax1.plot([mn,mx],[mn-0.5,mx-0.5],'--',color='gray',lw=1,alpha=0.5)
ax1.set_xlabel('Actual Rating'); ax1.set_ylabel('Predicted Rating')
ax1.set_title('Predicted vs Actual (colored by |error|)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=8)
ax1.text(0.05, 0.92, f'R² = {r2:.4f}\nRMSE = {rmse:.4f}',
         transform=ax1.transAxes, fontsize=10, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor=PALETTE['sky_l'], alpha=0.9))

# ── Residual vs Predicted
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(y_pred_reg[:5000], residuals[:5000],
            alpha=0.25, s=10, color=PALETTE['sky'], edgecolors='none')
ax2.axhline(0, color='red', lw=2, ls='--')
ax2.axhline(residuals.std(), color='gray', lw=1, ls=':', label=f'+1σ={residuals.std():.3f}')
ax2.axhline(-residuals.std(), color='gray', lw=1, ls=':',label=f'-1σ')
ax2.set_xlabel('Predicted Rating'); ax2.set_ylabel('Residual')
ax2.set_title('Residual Plot (Homoscedasticity Check)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)

# ── Residual Histogram + KDE
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(residuals, bins=50, color=PALETTE['sky_l'],
         edgecolor=PALETTE['sky'], lw=0.6, density=True)
kde_x = np.linspace(residuals.min(), residuals.max(), 300)
kde   = scipy_stats.gaussian_kde(residuals, bw_method=0.3)
ax3.plot(kde_x, kde(kde_x), color=PALETTE['sky_d'], lw=2.5, label='KDE')
x_norm = np.linspace(residuals.min(), residuals.max(), 300)
ax3.plot(x_norm, scipy_stats.norm.pdf(x_norm, residuals.mean(), residuals.std()),
         color=PALETTE['sand'], lw=2, ls='--', label='Normal fit')
ax3.axvline(0, color='red', lw=1.5, ls='--')
ax3.set_xlabel('Residual'); ax3.set_ylabel('Density')
ax3.set_title(f'Residual Distribution\nSkew={scipy_stats.skew(residuals):.3f}, Kurt={scipy_stats.kurtosis(residuals):.3f}',
              fontsize=12, fontweight='bold')
ax3.legend(fontsize=9)

# ── Error by rating bucket
ax4 = fig.add_subplot(gs[1, 0])
buckets = pd.cut(y_test_r, bins=[2.9,3.5,4.0,4.5,5.01],
                  labels=['3.0-3.5','3.5-4.0','4.0-4.5','4.5-5.0'])
err_df  = pd.DataFrame({'residual':residuals,'bucket':buckets})
bucket_stats = err_df.groupby('bucket',observed=True)['residual'].agg(
    ['mean','std',lambda x: np.sqrt(np.mean(x**2))]).reset_index()
bucket_stats.columns = ['bucket','bias','std','rmse']
x_pos = range(len(bucket_stats))
ax4.bar([i-0.2 for i in x_pos], bucket_stats['bias'], 0.35,
        label='Bias (mean residual)', color=PALETTE['sky'], edgecolor='white')
ax4.bar([i+0.2 for i in x_pos], bucket_stats['rmse'], 0.35,
        label='RMSE', color=PALETTE['sand'], edgecolor='white')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(bucket_stats['bucket'], fontsize=9)
ax4.axhline(0, color='gray', lw=1); ax4.set_title('Bias & RMSE by Rating Bucket', fontsize=12, fontweight='bold')
ax4.legend(fontsize=9); ax4.set_ylabel('Error')

# ── Q-Q Plot
ax5 = fig.add_subplot(gs[1, 1])
scipy_stats.probplot(residuals[:3000], dist='norm', plot=ax5)
ax5.get_lines()[0].set(color=PALETTE['sky'], markersize=2, alpha=0.4)
ax5.get_lines()[1].set(color=PALETTE['sand'], linewidth=2)
ax5.set_title('Q-Q Plot — Residual Normality', fontsize=12, fontweight='bold')

# ── Feature Importance
ax6 = fig.add_subplot(gs[1, 2])
fi_reg = pd.Series(reg.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
colors_reg = [PALETTE['mint'] if v >= fi_reg.quantile(0.75) else '#d1fae5' for v in fi_reg.values]
ax6.barh(range(len(fi_reg)), fi_reg.values, color=colors_reg, edgecolor='white')
ax6.set_yticks(range(len(fi_reg)))
ax6.set_yticklabels(fi_reg.index, fontsize=8.5)
ax6.set_xlabel('Feature Importance'); ax6.set_title('GBR Feature Importances', fontsize=12, fontweight='bold')
for i, v in enumerate(fi_reg.values):
    ax6.text(v+0.0003, i, f'{v:.4f}', va='center', fontsize=8)

plt.suptitle('Regression Model — Comprehensive Evaluation Dashboard',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('../static/charts/nb4_reg_evaluation.png', bbox_inches='tight', dpi=130)
plt.show()


## 3. Recommender System — Qualitative & Quantitative Evaluation

In [ ]:
# ─── 4.1 LOAD RECOMMENDERS & WARM UP ─────────────────────────────────────────
hybrid   = models.get('hybrid_recommender')
content  = models.get('content_recommender')
pop_rec  = models.get('popularity_recommender')
act_rec  = models.get('activity_recommender')

print("✓ Recommenders loaded:")
print(f"  Hybrid   : {type(hybrid).__name__}")
print(f"  Content  : {type(content).__name__}")
print(f"  Popularity: {type(pop_rec).__name__}")
print(f"  Activity : {type(act_rec).__name__}")
print(f"\nHybrid weights: {hybrid.weights}")


In [ ]:
# ─── 4.2 QUALITATIVE EVALUATION — 8 DIVERSE USE CASES ────────────────────────
use_cases = [
    {
        'id': 'UC-01',
        'label': 'Budget Backpacker — Spring Nature',
        'params': {'category':'Natural Scenery','season':'Spring','budget_level':'Budget','min_rating':4.0,'n':5},
        'expected': 'Low-cost natural scenic spots; high rating; Spring season match',
    },
    {
        'id': 'UC-02',
        'label': 'Premium Culture Seeker — Autumn Heritage',
        'params': {'category':'Historical Culture','season':'Autumn','budget_level':'Premium','min_rating':4.3,'n':5},
        'expected': 'High-end historical sites; 5A preferred; Autumn season match',
    },
    {
        'id': 'UC-03',
        'label': 'Family Group — Summer Theme Park',
        'params': {'category':'Theme Park','season':'Summer','min_rating':3.8,'n':5},
        'expected': 'Family-friendly theme parks; high popularity; summer suitability',
    },
    {
        'id': 'UC-04',
        'label': 'Spiritual Traveler — Winter Religious Sites',
        'params': {'category':'Religious Culture','season':'Winter','min_rating':4.0,'n':5},
        'expected': 'Temples and shrines; peaceful winter visits; moderate budget',
    },
    {
        'id': 'UC-05',
        'label': 'Destination-to-Destination: Similar to Xi Hu',
        'params': {'attraction_name':'Xi Hu','n':5},
        'expected': 'Similar lake/urban scenic areas; comparable rating and category',
    },
    {
        'id': 'UC-06',
        'label': 'Ancient Town Explorer — Any Season',
        'params': {'category':'Ancient Town','budget_level':'Mid-Range','min_rating':4.1,'n':5},
        'expected': 'Well-preserved ancient towns; cultural heritage; moderate price',
    },
    {
        'id': 'UC-07',
        'label': 'Activity-Based: Hiking/Mountain',
        'params_activity': 'mountain',
        'expected': 'Mountain and hiking destinations; natural scenery category dominant',
    },
    {
        'id': 'UC-08',
        'label': 'Luxury Autumn Experience',
        'params': {'season':'Autumn','budget_level':'Luxury','min_rating':4.5,'n':5},
        'expected': 'Premium attractions; very high rating; autumn foliage destinations',
    },
]

print("=" * 80)
print("QUALITATIVE EVALUATION — 8 USE CASES")
print("=" * 80)

qualitative_records = []
for uc in use_cases:
    print(f"\n{'─'*78}")
    print(f"  [{uc['id']}] {uc['label']}")
    print(f"  Expected: {uc['expected']}")
    print(f"{'─'*78}")

    if 'params_activity' in uc:
        result = act_rec.recommend_by_activity(uc['params_activity'], n=5)
    else:
        result = hybrid.recommend(**uc['params'])

    if result.empty:
        print("  [No results — query too restrictive]")
        continue

    # Display results
    for rank, (_, row) in enumerate(result.iterrows(), 1):
        name   = row.get('attraction_name', 'N/A')
        prov   = row.get('province', 'N/A')
        cat    = row.get('category', 'N/A')
        rating = row.get('avg_rating', 0)
        budget = row.get('budget_level', 'N/A')
        season = row.get('best_season', 'N/A')
        score  = row.get('hybrid_score', row.get('activity_score', row.get('popularity_score', 0)))
        level  = row.get('attraction_level', '')
        print(f"  #{rank} {name:<35} ★{rating:.2f} | {cat[:20]:<20} | {budget:<10} | {season:<7} | {level}")

    qualitative_records.append({
        'use_case': uc['id'],
        'label': uc['label'],
        'n_results': len(result),
        'avg_rating': result['avg_rating'].mean() if 'avg_rating' in result.columns else 0,
        'category_match': (result['category'] == uc.get('params',{}).get('category','')).mean()
                          if 'category' in result.columns else 'N/A',
        'budget_match': (result['budget_level'].astype(str) == uc.get('params',{}).get('budget_level','')).mean()
                        if 'budget_level' in result.columns else 'N/A',
    })

print("\n" + "=" * 80)


In [ ]:
# ─── 4.3 QUANTITATIVE EVALUATION — PRECISION@K, DIVERSITY, COVERAGE ──────────
print("=" * 70)
print("RECOMMENDATION SYSTEM — QUANTITATIVE EVALUATION")
print("=" * 70)

categories_all = catalog['category'].value_counts().head(12).index.tolist()
seasons_all    = ['Spring','Summer','Autumn','Winter']
top_pop        = catalog.nlargest(int(len(catalog)*0.20),'popularity_score')['attraction_name'].tolist()

# Metrics containers
quality_all, diversity_all, novelty_all = [], [], []
precision_at5_all, precision_at10_all   = [], []
catalog_coverage = set()

# Evaluate across 48 diverse queries
for cat in categories_all:
    for season in seasons_all:
        try:
            r = hybrid.recommend(category=cat, season=season, n=10, min_rating=3.5)
            if r.empty: continue

            # Quality: avg rating of results
            quality_all.append(r['avg_rating'].mean())

            # Diversity: proportion of unique categories in results
            cats_in_r = r['category'].tolist()
            diversity_all.append(len(set(cats_in_r)) / len(cats_in_r))

            # Novelty: fraction of results NOT in top-20% popular
            novelty_all.append((~r['attraction_name'].isin(top_pop)).mean())

            # Precision@5 and @10 (category relevance)
            p5  = (r.head(5)['category']  == cat).mean()
            p10 = (r.head(10)['category'] == cat).mean()
            precision_at5_all.append(p5)
            precision_at10_all.append(p10)

            catalog_coverage.update(r['attraction_name'].tolist())
        except Exception:
            pass

coverage_pct = len(catalog_coverage) / len(catalog)

print(f"\n  Queries evaluated          : {len(quality_all)}")
print(f"  Attractions surfaced       : {len(catalog_coverage)} / {len(catalog)} ({coverage_pct*100:.1f}%)")
print(f"\n  Quality (Avg Rating):")
print(f"    Mean ± Std             : {np.mean(quality_all):.4f} ± {np.std(quality_all):.4f}")
print(f"    Min / Max              : {np.min(quality_all):.4f} / {np.max(quality_all):.4f}")
print(f"\n  Precision@K (Category Match):")
print(f"    Precision@5            : {np.mean(precision_at5_all):.4f} ± {np.std(precision_at5_all):.4f}")
print(f"    Precision@10           : {np.mean(precision_at10_all):.4f} ± {np.std(precision_at10_all):.4f}")
print(f"\n  Diversity (Intra-list):")
print(f"    Mean ± Std             : {np.mean(diversity_all):.4f} ± {np.std(diversity_all):.4f}")
print(f"\n  Novelty (Non-popular fraction):")
print(f"    Mean ± Std             : {np.mean(novelty_all):.4f} ± {np.std(novelty_all):.4f}")
print(f"\n  Coverage                   : {coverage_pct*100:.2f}%")
print("=" * 70)


In [ ]:
# ─── 4.4 SYSTEM COMPARISON BENCHMARK ─────────────────────────────────────────
# Compare Popularity vs Content vs Hybrid across the same query set
benchmark_queries = [
    {'category': cat, 'n': 10}
    for cat in catalog['category'].value_counts().head(8).index.tolist()
]

systems = {
    'Popularity Baseline': {'rec': pop_rec, 'score_col': 'popularity_score', 'color': PALETTE['sand']},
    'Content-Based':       {'rec': content,  'score_col': 'similarity_score', 'color': PALETTE['mint']},
    'Hybrid (Production)': {'rec': hybrid,   'score_col': 'hybrid_score',     'color': PALETTE['sky']},
}

bench_results = {name: {'quality':[],'diversity':[],'novelty':[]} for name in systems}

for q in benchmark_queries:
    cat = q['category']
    for name, sys_info in systems.items():
        try:
            if name == 'Popularity Baseline':
                r = sys_info['rec'].recommend(category=cat, n=10)
            elif name == 'Content-Based':
                r = sys_info['rec'].search_by_query(cat, n=10)
            else:
                r = sys_info['rec'].recommend(category=cat, n=10)
            if r.empty: continue
            bench_results[name]['quality'].append(r['avg_rating'].mean())
            cats_r = r['category'].tolist()
            bench_results[name]['diversity'].append(len(set(cats_r))/len(cats_r))
            bench_results[name]['novelty'].append((~r['attraction_name'].isin(top_pop)).mean())
        except Exception:
            pass

# Aggregate
bench_agg = {}
for name in systems:
    br = bench_results[name]
    bench_agg[name] = {
        'quality':   np.mean(br['quality'])   if br['quality']   else 0,
        'diversity': np.mean(br['diversity']) if br['diversity'] else 0,
        'novelty':   np.mean(br['novelty'])   if br['novelty']   else 0,
    }

# Print table
print("SYSTEM BENCHMARK — 8 Category Queries")
print(f"{'System':<25} {'Quality':>9} {'Diversity':>10} {'Novelty':>9}")
print("─" * 58)
for name, vals in bench_agg.items():
    marker = " ← BEST" if name == 'Hybrid (Production)' else ""
    print(f"  {name:<23} {vals['quality']:>9.4f} {vals['diversity']:>10.4f} {vals['novelty']:>9.4f}{marker}")

# Visualise benchmark
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics_bench = [('quality','Avg Rating of Results','higher = better quality'),
                 ('diversity','Intra-List Diversity','higher = more varied'),
                 ('novelty','Novelty Score','higher = less mainstream')]

for ax, (metric, title, subtitle) in zip(axes, metrics_bench):
    vals   = [bench_agg[n][metric] for n in systems]
    colors = [sys_info['color'] for sys_info in systems.values()]
    bars   = ax.bar(range(len(systems)), vals, color=colors, edgecolor='white', width=0.6)
    ax.set_xticks(range(len(systems)))
    ax.set_xticklabels(list(systems.keys()), rotation=15, ha='right', fontsize=9)
    ax.set_title(f'{title}\n({subtitle})', fontsize=11, fontweight='bold')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.002,
                f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_ylim(0, max(vals)*1.18)

plt.suptitle('System Benchmark: Popularity vs Content-Based vs Hybrid',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/charts/nb4_system_benchmark.png', bbox_inches='tight', dpi=130)
plt.show()


## 4. Clustering — Final Validation

In [ ]:
# ─── 5.1 CLUSTERING QUALITY METRICS ──────────────────────────────────────────
kmeans   = models.get('clustering_model')
c_scaler = models.get('cluster_scaler')

NUM_FEATS = ['avg_rating','avg_ticket_price','avg_spend','avg_duration',
             'popularity_score','visit_count','highly_recommended_pct','very_satisfied_pct']
NUM_FEATS = [f for f in NUM_FEATS if f in catalog.columns]

X_clust  = catalog[NUM_FEATS].fillna(0).values
X_scaled = c_scaler.transform(X_clust)
labels   = kmeans.predict(X_scaled)
catalog_cl = catalog.copy()
catalog_cl['cluster'] = labels

# Clustering metrics
sil  = silhouette_score(X_scaled, labels)
db   = davies_bouldin_score(X_scaled, labels)
ch   = calinski_harabasz_score(X_scaled, labels)

print("=" * 55)
print("  CLUSTERING — FINAL VALIDATION METRICS")
print("=" * 55)
print(f"  K (clusters)        : {kmeans.n_clusters}")
print(f"  Silhouette Score    : {sil:.4f}  (higher = better, max=1)")
print(f"  Davies-Bouldin      : {db:.4f}   (lower  = better)")
print(f"  Calinski-Harabasz   : {ch:.1f}  (higher = better)")
print(f"  Inertia (WCSS)      : {kmeans.inertia_:.2f}")
print("=" * 55)
print("\nCluster Sizes:")
for k in range(kmeans.n_clusters):
    n = (labels == k).sum()
    bar = '█' * (n // 5)
    print(f"  Cluster {k}: {n:3d} attractions  {bar}")


In [ ]:
# ─── 5.2 CLUSTERING DASHBOARD ────────────────────────────────────────────────
CLUSTER_NAMES = {
    0:'Budget Cultural Explorers', 1:'Luxury Nature Seekers',
    2:'Family Heritage Travelers', 3:'Adventure Enthusiasts',
    4:'Urban Experience Seekers',  5:'Spiritual & Wellness Tourists',
}

fig = plt.figure(figsize=(22, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# PCA 2D
ax1 = fig.add_subplot(gs[0, :2])
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
for k in range(kmeans.n_clusters):
    mask = labels == k
    ax1.scatter(X_pca[mask,0], X_pca[mask,1], c=[CHART_COLORS[k]], s=80,
                alpha=0.75, edgecolors='white', linewidths=0.8,
                label=f"C{k}: {CLUSTER_NAMES.get(k,'Cluster '+str(k))}")
# Centroids
centers = pca.transform(kmeans.cluster_centers_)
ax1.scatter(centers[:,0], centers[:,1], c='white', marker='*', s=400,
            zorder=10, edgecolors='black', linewidths=1.5, label='Centroids')
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax1.set_title(f'KMeans Clustering — PCA Projection (K={kmeans.n_clusters})
'
              f'Silhouette={sil:.4f} | DB={db:.4f} | CH={ch:.1f}', fontsize=12, fontweight='bold')
ax1.legend(fontsize=8, loc='upper right', ncol=2)

# Cluster size pie
ax2 = fig.add_subplot(gs[0, 2])
sizes  = [(labels==k).sum() for k in range(kmeans.n_clusters)]
names  = [CLUSTER_NAMES.get(k,f'C{k}') for k in range(kmeans.n_clusters)]
wedges, texts, autos = ax2.pie(sizes, labels=None, autopct='%1.1f%%',
    colors=CHART_COLORS[:kmeans.n_clusters], wedgeprops=dict(width=0.55, edgecolor='white', lw=1.5))
ax2.legend(wedges, [f'C{k}: {n}' for k,n in enumerate(names)],
           loc='lower center', fontsize=7, ncol=2, bbox_to_anchor=(0.5,-0.15))
ax2.set_title('Cluster Size Distribution', fontsize=12, fontweight='bold')

# Radar chart (cluster profiles)
ax3 = fig.add_subplot(gs[1, 0])
metrics_radar = ['avg_rating','avg_ticket_price','avg_spend','popularity_score','highly_recommended_pct']
metrics_radar = [m for m in metrics_radar if m in catalog.columns]
cluster_means = catalog_cl.groupby('cluster')[metrics_radar].mean()
cluster_norm  = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min() + 1e-9)
for k in range(kmeans.n_clusters):
    ax3.bar(np.arange(len(metrics_radar)) + k*0.12, cluster_norm.iloc[k].values,
            width=0.12, color=CHART_COLORS[k], alpha=0.8, label=f'C{k}')
ax3.set_xticks(np.arange(len(metrics_radar)) + 0.3)
ax3.set_xticklabels([m.replace('_',' ')[:12] for m in metrics_radar],
                     rotation=25, ha='right', fontsize=8)
ax3.set_title('Cluster Feature Profiles (Normalized)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Normalized Value')
ax3.legend(fontsize=7, ncol=3, loc='upper right')

# Category distribution per cluster
ax4 = fig.add_subplot(gs[1, 1:])
top_cats_c = catalog['category'].value_counts().head(8).index.tolist()
cluster_cat = pd.crosstab(catalog_cl['cluster'], catalog_cl['category'])[top_cats_c]
cluster_cat_pct = cluster_cat.div(cluster_cat.sum(axis=1), axis=0) * 100
cluster_cat_pct.index = [CLUSTER_NAMES.get(k,f'C{k}') for k in cluster_cat_pct.index]
cluster_cat_pct.plot(kind='bar', ax=ax4, color=CHART_COLORS[:8],
                     edgecolor='white', width=0.8)
ax4.set_title('Category Mix by Cluster (%)', fontsize=12, fontweight='bold')
ax4.set_ylabel('%'); ax4.set_xticklabels(ax4.get_xticklabels(), rotation=20, ha='right', fontsize=8.5)
ax4.legend(title='Category', fontsize=7, bbox_to_anchor=(1.01,1), loc='upper left')

plt.suptitle('Clustering Model — Comprehensive Evaluation Dashboard',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('../static/charts/nb4_clustering.png', bbox_inches='tight', dpi=130)
plt.show()


## 5. Model Comparison Dashboard

In [ ]:
# ─── 6.1 FINAL MODEL COMPARISON TABLE ────────────────────────────────────────
model_comparison = pd.DataFrame([
    {
        'Model':            'Popularity Recommender',
        'Type':             'Recommendation',
        'Algorithm':        'Weighted Score',
        'Primary Metric':   'Popularity Score',
        'Score':            f'{np.mean(bench_agg["Popularity Baseline"]["quality"]):.4f}',
        'Status':           'Baseline',
        'Use Case':         'Cold-start, broad queries',
        'Training Time':    '<1s',
        'Inference Time':   'O(n) catalog scan',
    },
    {
        'Model':            'Content-Based (TF-IDF)',
        'Type':             'Recommendation',
        'Algorithm':        'TF-IDF + Cosine + KNN',
        'Primary Metric':   'Avg Result Rating',
        'Score':            f'{np.mean(bench_agg["Content-Based"]["quality"]):.4f}',
        'Status':           'Production',
        'Use Case':         'Destination similarity, query matching',
        'Training Time':    '~5s',
        'Inference Time':   'O(k) KNN lookup',
    },
    {
        'Model':            'Hybrid Recommender',
        'Type':             'Recommendation',
        'Algorithm':        '6-Signal Weighted Ensemble',
        'Primary Metric':   'Composite Score',
        'Score':            f'{np.mean(quality_all):.4f}',
        'Status':           '★ Production (Primary)',
        'Use Case':         'Full personalized recommendations',
        'Training Time':    '<1s (uses sub-models)',
        'Inference Time':   'O(n) catalog + O(k) KNN',
    },
    {
        'Model':            'Activity Recommender',
        'Type':             'Recommendation',
        'Algorithm':        'Keyword Matching + Score',
        'Primary Metric':   'Activity Match Rate',
        'Score':            'N/A',
        'Status':           'Production',
        'Use Case':         'Activity-type destination search',
        'Training Time':    '<1s',
        'Inference Time':   'O(n) keyword scan',
    },
    {
        'Model':            'Random Forest Classifier',
        'Type':             'Classification',
        'Algorithm':        'Random Forest (200 trees)',
        'Primary Metric':   'F1-Weighted',
        'Score':            f'{f1_w:.4f}',
        'Status':           'Production',
        'Use Case':         'Predict recommendation level',
        'Training Time':    '~45s',
        'Inference Time':   'O(depth × trees)',
    },
    {
        'Model':            'Gradient Boosting Regressor',
        'Type':             'Regression',
        'Algorithm':        'GBR (200 estimators)',
        'Primary Metric':   'R² Score',
        'Score':            f'{r2:.4f}',
        'Status':           'Production',
        'Use Case':         'Predict attraction rating',
        'Training Time':    '~60s',
        'Inference Time':   'O(depth × estimators)',
    },
    {
        'Model':            'KMeans Clustering',
        'Type':             'Clustering',
        'Algorithm':        'KMeans (k=6)',
        'Primary Metric':   'Silhouette Score',
        'Score':            f'{sil:.4f}',
        'Status':           'Production',
        'Use Case':         'Traveler segment profiling',
        'Training Time':    '~3s',
        'Inference Time':   'O(k×d) centroid distance',
    },
])

display(model_comparison.style
    .applymap(lambda v: 'background-color: #d1fae5; font-weight: bold'
              if '★' in str(v) or 'Production' in str(v) else '', subset=['Status'])
    .set_properties(**{'font-size':'11.5px'})
    .set_table_styles([{'selector':'th','props':[('background-color','#e0f2fe'),
                                                  ('font-weight','bold'),
                                                  ('font-size','12px')]}]))


In [ ]:
# ─── 6.2 SUMMARY DASHBOARD VISUALIZATION ─────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(21, 12))

# ── ML Metrics Bar Chart
ax1 = axes[0, 0]
ml_metrics = {
    'Clf Accuracy':   acc,
    'Clf F1-W':       f1_w,
    'Clf Precision':  prec,
    'Clf Recall':     rec,
    'Reg R²':         r2,
    'Clustering Sil': sil,
}
bar_colors = [PALETTE['sky'] if 'Clf' in k else
              PALETTE['mint'] if 'Reg' in k else PALETTE['sand'] for k in ml_metrics]
bars = ax1.bar(range(len(ml_metrics)), list(ml_metrics.values()),
               color=bar_colors, edgecolor='white')
ax1.set_xticks(range(len(ml_metrics)))
ax1.set_xticklabels(list(ml_metrics.keys()), rotation=25, ha='right', fontsize=9)
ax1.set_ylim(0, 1.15)
ax1.set_title('ML Model Final Metrics Summary', fontsize=12, fontweight='bold')
ax1.set_ylabel('Score')
for bar, v in zip(bars, ml_metrics.values()):
    ax1.text(bar.get_x()+bar.get_width()/2, v+0.01,
             f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
legend_patches = [mpatches.Patch(color=PALETTE['sky'],  label='Classification'),
                  mpatches.Patch(color=PALETTE['mint'], label='Regression'),
                  mpatches.Patch(color=PALETTE['sand'], label='Clustering')]
ax1.legend(handles=legend_patches, fontsize=9)

# ── Recommender Quality Comparison
ax2 = axes[0, 1]
sys_labels = list(bench_agg.keys())
sys_quality = [bench_agg[s]['quality'] for s in sys_labels]
sys_colors  = [PALETTE['sand'], PALETTE['mint'], PALETTE['sky']]
bars2 = ax2.bar(range(len(sys_labels)), sys_quality, color=sys_colors, edgecolor='white')
ax2.set_xticks(range(len(sys_labels)))
ax2.set_xticklabels(sys_labels, rotation=15, ha='right', fontsize=9)
ax2.set_title('Recommender: Avg Result Quality', fontsize=12, fontweight='bold')
ax2.set_ylabel('Avg Rating of Results')
ax2.set_ylim(0, 5.5)
for bar, v in zip(bars2, sys_quality):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.05, f'{v:.4f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# ── Precision@K by Category
ax3 = axes[0, 2]
p5_by_cat  = [np.mean([precision_at5_all[i] for i, q in enumerate(
    [(c,s) for c in categories_all for s in seasons_all]) if q[0]==cat])
    for cat in categories_all[:8]]
ax3.barh(categories_all[:8], p5_by_cat, color=CHART_COLORS[:8], edgecolor='white')
ax3.set_xlabel('Precision@5 (Category Match Rate)')
ax3.set_title('Precision@5 by Category', fontsize=12, fontweight='bold')
ax3.set_xlim(0, 1.15)
for i, v in enumerate(p5_by_cat):
    ax3.text(v+0.01, i, f'{v:.3f}', va='center', fontsize=9)

# ── CV Score Stability
ax4 = axes[1, 0]
cv_clf_f1  = cross_val_score(clf, X, y_clf, cv=skf, scoring='f1_weighted', n_jobs=-1)
cv_reg_r2  = cross_val_score(reg, X, y_reg, cv=kf,  scoring='r2',          n_jobs=-1)
ax4.plot(range(1, len(cv_clf_f1)+1), cv_clf_f1, 'o-', color=PALETTE['sky'],
         lw=2.5, markersize=8, markerfacecolor='white', markeredgewidth=2, label='Clf F1-W')
ax4.plot(range(1, len(cv_reg_r2)+1), cv_reg_r2, 's-', color=PALETTE['mint'],
         lw=2.5, markersize=8, markerfacecolor='white', markeredgewidth=2, label='Reg R²')
ax4.axhline(cv_clf_f1.mean(), color=PALETTE['sky'], ls='--', lw=1.5, alpha=0.6)
ax4.axhline(cv_reg_r2.mean(), color=PALETTE['mint'], ls='--', lw=1.5, alpha=0.6)
ax4.set_xlabel('CV Fold'); ax4.set_ylabel('Score')
ax4.set_title('5-Fold CV Score Stability', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10); ax4.grid(alpha=0.3)
ax4.set_xticks(range(1, 6))

# ── Recommender Metric Radar (as grouped bar)
ax5 = axes[1, 1]
rec_metrics_names = ['Quality
(norm)', 'Diversity', 'Novelty', 'Coverage']
rec_metrics_vals  = {
    'Popularity': [bench_agg['Popularity Baseline']['quality']/5,
                   bench_agg['Popularity Baseline']['diversity'],
                   bench_agg['Popularity Baseline']['novelty'], 0.15],
    'Content':    [bench_agg['Content-Based']['quality']/5,
                   bench_agg['Content-Based']['diversity'],
                   bench_agg['Content-Based']['novelty'], 0.25],
    'Hybrid':     [bench_agg['Hybrid (Production)']['quality']/5,
                   bench_agg['Hybrid (Production)']['diversity'],
                   bench_agg['Hybrid (Production)']['novelty'], coverage_pct],
}
x5 = np.arange(len(rec_metrics_names))
w5 = 0.25
for i, (name, vals) in enumerate(rec_metrics_vals.items()):
    color = [PALETTE['sand'], PALETTE['mint'], PALETTE['sky']][i]
    ax5.bar(x5 + (i-1)*w5, vals, w5, color=color, label=name, edgecolor='white', alpha=0.85)
ax5.set_xticks(x5); ax5.set_xticklabels(rec_metrics_names, fontsize=9)
ax5.set_ylim(0, 1.1); ax5.set_title('Recommender Multi-Metric Comparison', fontsize=12, fontweight='bold')
ax5.legend(fontsize=9); ax5.set_ylabel('Score (normalized)')

# ── Model Complexity vs Performance
ax6 = axes[1, 2]
model_info = {
    'Popularity
Baseline':  (0.05, bench_agg['Popularity Baseline']['quality'], 100),
    'Content
Based':        (0.3,  bench_agg['Content-Based']['quality'],       500),
    'Hybrid
Recommender':   (0.5,  bench_agg['Hybrid (Production)']['quality'], 900),
    'RF
Classifier':        (0.8,  f1_w * 5, 1500),
    'GBR
Regressor':        (0.7,  r2 * 5,   800),
    'KMeans
Clustering':    (0.2,  sil * 5,  300),
}
for name, (complexity, perf, size) in model_info.items():
    ax6.scatter(complexity, perf, s=size, alpha=0.75, edgecolors='white', linewidths=1.5,
                c=[CHART_COLORS[list(model_info.keys()).index(name) % len(CHART_COLORS)]])
    ax6.annotate(name, (complexity, perf), fontsize=8, ha='center', va='bottom',
                 xytext=(0, 8), textcoords='offset points')
ax6.set_xlabel('Model Complexity (relative)'); ax6.set_ylabel('Performance (scaled)')
ax6.set_title('Complexity vs Performance Trade-off
(Bubble size = training records)',
              fontsize=11, fontweight='bold')
ax6.grid(alpha=0.3)

plt.suptitle('TourMate AI — Final Model Evaluation Dashboard',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('../static/charts/nb4_final_dashboard.png', bbox_inches='tight', dpi=130)
plt.show()


## 6. Export All Production Artifacts

In [ ]:
# ─── 7.1 VERIFY ALL REQUIRED FILES ────────────────────────────────────────────
required_files = [
    'hybrid_recommender.pkl',
    'content_recommender.pkl',
    'popularity_recommender.pkl',
    'activity_recommender.pkl',
    'tfidf_vectorizer.pkl',
    'similarity_model.pkl',
    'classification_model.pkl',
    'regression_model.pkl',
    'clustering_model.pkl',
    'cluster_scaler.pkl',
    'clf_encoders.pkl',
    'attraction_catalog.pkl',
    'catalog.pkl',
    'model_metrics.json',
    'model_metadata.json',
    'feature_names.json',
]

print("=" * 60)
print("  PRODUCTION ARTIFACT VERIFICATION")
print("=" * 60)
total_size = 0
for fname in required_files:
    path = os.path.join(MODELS_DIR, fname)
    if os.path.exists(path):
        size = os.path.getsize(path)
        total_size += size
        print(f"  ✅ {fname:<40} {size/1024:>8.1f} KB")
    else:
        print(f"  ❌ MISSING: {fname}")

print(f"{'─'*60}")
print(f"  Total model artifacts: {total_size/1024/1024:.2f} MB")
print("=" * 60)


In [ ]:
# ─── 7.2 UPDATE FINAL METRICS JSON ───────────────────────────────────────────
final_metrics = {
    'classification': {
        'model':              'RandomForestClassifier',
        'n_estimators':       200,
        'test_accuracy':      round(float(acc),  6),
        'test_f1_weighted':   round(float(f1_w), 6),
        'test_f1_macro':      round(float(f1_m), 6),
        'test_precision_w':   round(float(prec), 6),
        'test_recall_w':      round(float(rec),  6),
        'cv_f1_mean':         round(float(cv_f1.mean()),  6),
        'cv_f1_std':          round(float(cv_f1.std()),   6),
        'cv_acc_mean':        round(float(cv_acc.mean()), 6),
        'target_classes':     le_target.classes_.tolist(),
        'n_features':         len(FEATURE_COLS),
        'feature_names':      FEATURE_COLS,
    },
    'regression': {
        'model':              'GradientBoostingRegressor',
        'n_estimators':       200,
        'test_r2':            round(float(r2),   6),
        'test_rmse':          round(float(rmse), 6),
        'test_mae':           round(float(mae),  6),
        'test_mape':          round(float(mape), 4),
        'within_05_pct':      round(float(within_05), 2),
        'within_10_pct':      round(float(within_10), 2),
        'cv_r2_mean':         round(float(cv_r2.mean()),   6),
        'cv_r2_std':          round(float(cv_r2.std()),    6),
        'cv_rmse_mean':       round(float(-cv_rmse.mean()),6),
        'target':             'rating',
        'residual_skew':      round(float(scipy_stats.skew(residuals)), 4),
        'residual_kurtosis':  round(float(scipy_stats.kurtosis(residuals)), 4),
    },
    'recommender': {
        'system':             'HybridRecommender',
        'n_signals':          6,
        'hybrid_weights':     hybrid.weights,
        'coverage_pct':       round(float(coverage_pct * 100), 2),
        'avg_quality':        round(float(np.mean(quality_all)), 4),
        'avg_precision_at5':  round(float(np.mean(precision_at5_all)), 4),
        'avg_precision_at10': round(float(np.mean(precision_at10_all)), 4),
        'avg_diversity':      round(float(np.mean(diversity_all)), 4),
        'avg_novelty':        round(float(np.mean(novelty_all)), 4),
        'n_attractions':      len(catalog),
        'tfidf_vocab_size':   len(content.tfidf.vocabulary_),
    },
    'clustering': {
        'algorithm':          'KMeans',
        'k':                  int(kmeans.n_clusters),
        'silhouette_score':   round(float(sil), 4),
        'davies_bouldin':     round(float(db),  4),
        'calinski_harabasz':  round(float(ch),  1),
        'inertia':            round(float(kmeans.inertia_), 2),
        'cluster_names':      CLUSTER_NAMES,
        'cluster_sizes':      {str(k): int((labels==k).sum())
                               for k in range(kmeans.n_clusters)},
        'pca_variance_explained': [round(float(v),4) for v in pca.explained_variance_ratio_],
    },
}

with open(os.path.join(MODELS_DIR, 'model_metrics.json'), 'w') as f:
    json.dump(final_metrics, f, indent=2)

print("✓ model_metrics.json updated with final evaluation metrics")
print(f"  Classification F1-W : {final_metrics['classification']['test_f1_weighted']}")
print(f"  Regression R²       : {final_metrics['regression']['test_r2']}")
print(f"  Recommender Coverage: {final_metrics['recommender']['coverage_pct']}%")
print(f"  Clustering Silhouette: {final_metrics['clustering']['silhouette_score']}")


## 7. System Architecture Summary

In [ ]:
# ─── 8. SYSTEM ARCHITECTURE SUMMARY ─────────────────────────────────────────
print("=" * 80)
print("  TOURMATE AI — PRODUCTION SYSTEM ARCHITECTURE SUMMARY")
print("=" * 80)

arch_summary = {
    "Data Layer": {
        "Source":              "tourism_recommendation_dataset_en.csv",
        "Records":             f"{len(df):,}",
        "Unique Attractions":  str(len(catalog)),
        "Provinces":           str(df['province'].nunique()),
        "Categories":          str(df['attraction_category'].nunique()),
    },
    "Feature Engineering": {
        "Budget Binning":      "5 levels: Free/Budget/Mid-Range/Premium/Luxury",
        "Popularity Score":    "0.4×freq_norm + 0.3×rating_norm + 0.2×highly_rec + 0.1×very_sat",
        "TF-IDF Features":     f"{len(content.tfidf.vocabulary_):,} terms (bigrams, sublinear_tf)",
        "Encoded Features":    f"{len(FEATURE_COLS)} features for ML models",
        "Combined Text":       "attraction_name + category + province + city + level + season + budget",
    },
    "Recommendation Engine": {
        "Popularity Baseline": "Weighted score: frequency × rating × satisfaction",
        "Content-Based":       f"TF-IDF cosine + NearestNeighbors (k=20), vocab={len(content.tfidf.vocabulary_):,}",
        "Hybrid System":       f"6-signal weighted: {hybrid.weights}",
        "Activity Search":     "Keyword matching on category + attraction name",
        "Gemini AI":           "Personalized travel insights, itineraries, tips",
    },
    "ML Models": {
        "Classifier":          f"Random Forest | F1-W={f1_w:.4f} | Acc={acc:.4f}",
        "Regressor":           f"Gradient Boosting | R²={r2:.4f} | RMSE={rmse:.4f}",
        "Clustering":          f"KMeans K=6 | Silhouette={sil:.4f} | 6 traveler profiles",
    },
    "Flask Application": {
        "Pages":               "Home, Recommend, Results, Explore, Activities, Dashboard, Model Perf, About",
        "API Endpoints":       "/api/recommend, /api/insight, /api/suggest, /api/stats",
        "UI Theme":            "Premium light — white, sky blue, sand, mint green",
        "Charts":              "11 interactive Plotly charts in EDA dashboard",
        "Model Loading":       "Single load at startup, cached in memory",
    },
}

for section, details in arch_summary.items():
    print(f"\n  ┌─ {section} {'─'*(60-len(section))}┐")
    for key, val in details.items():
        print(f"  │  {key:<25}: {val}")
    print(f"  └{'─'*63}┘")

print("\n" + "=" * 80)
print("  ✓ All notebooks complete. Run 'python app.py' to launch TourMate AI.")
print("=" * 80)
